# Source Separation Result

> Standardized result DTO for the source-separation (audio-preprocessing) task — the data noun source-separation tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

**Canonical home (execution stage 8 — Option C / PILLAR 1c):** `SourceSeparationResult` lives here in `cjm-capability-primitives` so a pure-compute source-separation tool capability (e.g. Demucs vocals isolation) depends only on this data noun, never on the adapter machinery (`TaskAdapter`, the tool protocol, persistence). It is the first member of the `source_separation` task family; the wire kind is `"source_separation.result"`.

**Artifact-producing shape (the family's distinguishing trait):** unlike the inline-data DTOs (transcription text / VAD ranges / FA items), a source-separation result's payload is an **audio file** — so the DTO carries `output_path` (the produced vocals-isolated artifact) plus a flat `metadata` dict, never the audio bytes. The fields are flat (a `str` + a `dict`), so the default wire reconstruction suffices — no custom `from_dict` (cf. `VADResult`/`ForcedAlignResult`, which re-type nested objects).

In [ ]:
#| default_exp source_separation

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict

from cjm_substrate.core.wire import wire_type

In [ ]:
#| export
@wire_type("source_separation.result")
@dataclass
class SourceSeparationResult:
    """Standardized output for source-separation (audio-preprocessing) capabilities.

    The payload is an AUDIO ARTIFACT, not inline data: `output_path` is the
    produced isolated-audio file (e.g. the vocals stem) the tool wrote to the
    location the adapter chose. `metadata` carries the stats the fused-era
    return dict held (duration, sample_rate, model, stems_available, and any
    extra-stem paths when the tool was asked to keep them)."""
    output_path: str                                        # Path to the produced isolated-audio artifact (e.g. vocals stem)
    metadata: Dict[str, Any] = field(default_factory=dict)  # Stats (duration, sample_rate, model, stems_available, other_stems, ...)

In [ ]:
# Test SourceSeparationResult
result = SourceSeparationResult(
    output_path="/data/cjm-media-plugin-demucs/separations/ab12cd_0f1e2d3c4b5a/vocals.wav",
    metadata={"duration": 300.0, "sample_rate": 44100, "model": "htdemucs",
              "stems_available": ["vocals", "drums", "bass", "other"]},
)

print(f"Output path: {result.output_path}")
print(f"Metadata: {result.metadata}")
assert result.output_path.endswith("vocals.wav")
assert result.metadata["model"] == "htdemucs"

In [ ]:
# Test minimal result (just the produced artifact path; empty metadata)
minimal = SourceSeparationResult(output_path="/tmp/vocals.wav")
print(f"Minimal result: {minimal}")
assert minimal.output_path == "/tmp/vocals.wav" and minimal.metadata == {}

In [ ]:
# Wire-format executable test: the result round-trips TYPED through the
# simulated worker boundary (encode -> json -> decode). The fields are flat
# (str + dict), so the default reconstruction rebuilds the typed object — no
# custom from_dict needed (cf. vad.result's nested TimeRanges).
import json as _json
from cjm_substrate.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

res = SourceSeparationResult(
    output_path="/data/sep/vocals.wav",
    metadata={"duration": 300.0, "sample_rate": 44100, "model": "htdemucs"},
)
envelope = wire_encode(res)
assert envelope[WIRE_KIND_KEY] == "source_separation.result"
back = wire_decode(_json.loads(_json.dumps(envelope)))
assert isinstance(back, SourceSeparationResult)
assert back == res
print("source_separation.result wire round-trip OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()